# 03 - Validacao de Pre-processamento e DataLoaders

Este notebook valida as tarefas 9 e 10 da ordem recomendada:

1. Pre-processamento das imagens.
2. Transforms de treino, validacao e teste.
3. Dataset PyTorch.
4. DataLoaders para treino, validacao e teste.

Ele depende dos arquivos `data/splits/train.csv`, `data/splits/val.csv` e `data/splits/test.csv`, gerados pelo notebook `02_preprocessamento_splits.ipynb`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.datasets import DataLoaderConfig, create_dataloaders, inspect_batch
from src.preprocessing import denormalize_tensor, get_eval_transforms, get_train_transforms

config.ensure_project_directories()
config.seed_everything()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", config.DEVICE)
print("Train split:", config.SPLITS_DIR / "train.csv")

## Validacao das Transforms

As transforms de treino aplicam augmentations leves. As transforms de validacao/teste sao deterministicas.

In [ ]:
train_transforms = get_train_transforms()
eval_transforms = get_eval_transforms()

print("Train transforms:")
print(train_transforms)
print("\nEval transforms:")
print(eval_transforms)

## Criacao dos DataLoaders

Esta celula falha com mensagem clara se os splits ainda nao existirem. Nesse caso, execute primeiro `02_preprocessamento_splits.ipynb`.

In [ ]:
required_splits = [config.SPLITS_DIR / f"{split}.csv" for split in ["train", "val", "test"]]
missing_splits = [path for path in required_splits if not path.exists()]

if missing_splits:
    raise FileNotFoundError(
        "Splits ausentes: "
        + ", ".join(str(path) for path in missing_splits)
        + ". Execute notebooks/02_preprocessamento_splits.ipynb primeiro."
    )

loader_config = DataLoaderConfig(batch_size=8, num_workers=0, pin_memory=torch.cuda.is_available())
loaders = create_dataloaders(dataloader_config=loader_config)

for split_name, loader in loaders.items():
    dataset = loader.dataset
    print(split_name, "imagens:", len(dataset), "class_counts:", dataset.class_counts)

## Inspecao de Batch

Validamos se as imagens chegam como tensores `[batch, 3, 224, 224]` e se os labels chegam como tensores numericos.

In [ ]:
batch = next(iter(loaders["train"]))
inspect_batch(batch)

## Visualizacao de Amostras Pre-processadas

As imagens foram normalizadas para modelos pre-treinados. Para visualizar, desfazemos a normalizacao.

In [ ]:
images, labels = batch
max_images = min(8, images.shape[0])

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.ravel()

for index in range(max_images):
    image = denormalize_tensor(images[index]).permute(1, 2, 0).cpu().numpy()
    label_idx = int(labels[index].item())
    axes[index].imshow(image)
    axes[index].set_title(config.INDEX_TO_CLASS[label_idx])
    axes[index].axis("off")

for ax in axes[max_images:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Proxima Etapa

Depois desta validacao, seguir para as tarefas 11 e 12:

1. Criar pipeline de treino.
2. Criar pipeline de avaliacao.